# Lab 7.6 &mdash; Draft: Generate a Grounded Reply

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 25 min &nbsp;|&nbsp; **Day 4 &middot; Module 7 &mdash; Task Automation with AI Agents**

### What you'll do
- Build a prompt grounded in the gathered order fields
- Call a REAL Groq model to draft the reply
- Check the draft is grounded -- it uses only fields you gave it

> **How this lab works (near-real):** you have a `GROQ_API_KEY` in the repo `.env`. Read the **Concept**, fill the real `___` blanks in **Build it** (real pipeline logic, real tool bodies, the real draft/`create_agent` call), then **Run it for real** &mdash; a real Groq model drives the step over real tools &mdash; and **read the output/trace**. Finish with an open **Your turn**. There is **no auto-grader**; the goal is a working email agent and a trace you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langchain-groq`) and a **real hosted model** (`ChatGroq("openai/gpt-oss-20b")` &mdash; reliable tool-calling via `create_agent`). If `GROQ_API_KEY` is unset, the run cells print how to set it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole agent run. You are building the **email-drafting agent** (the client's Lab 4.1): it **drafts but never sends** &mdash; `send_email` is withheld and a human approves.

**Reference:** [Module 7 slides &mdash; Draft — generate a work product](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 7 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, time, socket, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY (+ other keys)

WORK = "/tmp/biaa-lab-07-06"
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is configured -- the 'Run it for real' cells self-skip if not."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-4 provider: a REAL hosted model with reliable tool-calling via create_agent.
MODEL = "openai/gpt-oss-20b"
llm = ChatGroq(model=MODEL, temperature=0) if groq_ready() else None

def with_backoff(fn, tries=4):
    """Run fn(); retry with backoff on Groq rate limits (HTTP 429). Other errors propagate."""
    last = None
    for attempt in range(tries):
        try:
            return fn()
        except Exception as e:
            last = e
            if "429" in str(e) or "rate limit" in str(e).lower() or "rate_limit" in str(e).lower():
                wait = 5 * (attempt + 1)
                print(f"(rate-limited -- retrying in {wait}s)")
                time.sleep(wait)
                continue
            raise
    raise last

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("Groq ready | model:", MODEL, "| WORK:", WORK)
else:
    print("GROQ_API_KEY not set -- add it to the repo .env (free key at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
**Draft** turns gathered context into a work product a human will review (deck slide 9): a reply, a
summary, a proposal. Three keys: **ground it** (feed the real order so it's specific, not generic),
**give it a voice** (tone, format, limits &mdash; set in the prompt), and **keep the human** (a draft
is a suggestion, not a sent message). The failure mode to avoid: a draft that **invents** a date or a
policy because it wasn't grounded. Here a **real Groq model** writes the draft &mdash; grounded in the
order fields you pass in.

In [ ]:
from langchain_core.prompts import PromptTemplate

# The email agent's context sources: a small orders DB and a set of reply templates.
ORDERS = {
    "4471": {"id": "4471", "name": "Priya", "status": "shipped",    "eta": "Friday",    "carrier": "BlueDart"},
    "5090": {"id": "5090", "name": "Sam",   "status": "processing", "eta": "next week", "carrier": "-"},
}
TEMPLATES = {
    "delivery_delay": "Hi {name}, your order {id} has {status} and is due {eta}. Thanks for your patience.",
    "refund":         "Hi {name}, we've started the refund for order {id}. It will reflect in 5-7 days.",
}

print("order to ground on:", ORDERS["4471"])

## Build it
Complete `build_prompt` (fill the grounded prompt from the order &mdash; never invent a field) and
`draft_reply` (call the real model on it).

In [ ]:
from langchain_core.prompts import PromptTemplate

DRAFT_PROMPT = PromptTemplate.from_template(
    "You are a customer-support agent. Using ONLY these order facts, write a short, polite reply.\n"
    "Invent no date or fact that is not given below.\n"
    "Customer name: {name}\nOrder id: {id}\nStatus: {status}\nETA: {eta}\nReply:")

def build_prompt(order):
    # ground the draft in the REAL order fields -- never invent anything
    return DRAFT_PROMPT.format(name=___, id=___, status=___, eta=___)   # TODO: from order

def draft_reply(order):
    """Ask the REAL Groq model to draft a reply, grounded in this order."""
    return ___   # TODO: with_backoff(lambda: llm.invoke(build_prompt(order)).content)

try:
    # No model call here -- just inspect the grounded prompt the model will receive:
    print(build_prompt(ORDERS["4471"]))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## Run it for real &amp; read the trace
Call the real Groq model through `draft_reply()` and read the reply it writes &mdash; grounded in the order you gave it.

_This calls the real `openai/gpt-oss-20b` via Groq. If `GROQ_API_KEY` is unset the cell prints how to set it instead of crashing._

In [ ]:
if not groq_ready():
    print("No GROQ_API_KEY -- add it to the repo .env (free key at console.groq.com), then re-run this cell.")
else:
    try:
        reply = draft_reply(ORDERS["4471"])
        print(reply)
        print("---")
        print("mentions the id? ", "4471" in reply)
        print("grounded ETA?    ", "Friday" in reply)
    except Exception as e:
        print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- The reply is written by the **real model**, yet it names the real order id and the real ETA (`Friday`) &mdash; because the prompt grounded it.
- Nothing in the prompt says "Friday" is optional; the instruction *"invent no date not given"* is the guardrail against a hallucinated promise.
- Change the ETA in `ORDERS["4471"]` and re-run: the draft follows the data. That's grounding, not luck.

## Your turn (open task &mdash; no grader)
Change the **voice** in `DRAFT_PROMPT` &mdash; make it warmer, add a sign-off, cap it at two sentences,
or draft in another language &mdash; and re-run `draft_reply()`. **What good looks like:** the style changes
with your prompt while the **facts stay grounded** (id and ETA still correct). Try feeding `ORDERS["5090"]`
(a different order) and confirm the draft tracks *its* fields.

---
### Key takeaway
A grounded draft is specific and correct; an ungrounded one invents facts. Draft agents are high-value and low-risk because the human still holds the pen -- which is why the email agent is the canonical first real-world lab.

[&#8592; All Module 7 labs](./index.html) &nbsp;&middot;&nbsp; [Module 7 slides](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>